# Understanding Mpox: Key Facts and Global Impact

Generated starter notebook from [DesiData](https://www.desidata.in) — India's open dataset catalogue. Run the cells top to bottom.

Mpox, previously known as Monkeypox, is a zoonotic viral disease with symptoms resembling Smallpox but with less clinical severity. The disease was first discovered in 1958 in colonies of monkeys kept for research, and the first human case was identified in 1970 in the Democratic Republic of the Congo (DRC). Since then, cases have been reported in 15 African countries, and the disease has spread globally, with 123 Member States reporting cases to WHO since January 2022. As of August 2024, a total of 106,310 laboratory-confirmed cases and 234 deaths have been reported worldwide. The majority of recent cases have been notified from the African Region (62.3%) and the European Region (13.7%). Mpox is caused by the monkeypox virus, which belongs to the genus Orthopoxvirus and family Poxviridae. There are two distinct genetic clades of the virus: the Central African (Congo Basin) clade, renamed Clade I, and the West African clade, renamed Clade II. Clade II is further divided into subclades IIa and IIb, with the latter associated with the global outbreak in 2022–2023. The disease continues to evolve, with genetic changes observed in the virus. Human-to-human transmission occurs through large respiratory droplets requiring prolonged close contact, as well as through indirect contact with contaminated materials. The incubation period ranges from 5 to 21 days, and a person is not contagious during this time. Severe cases are more common among children and can lead to complications such as secondary bacterial infections, dehydration, pneumonia, and encephalitis. Surveillance and preventive measures, including isolation, contact tracing, and vaccination for high-risk individuals, are critical in controlling the spread of Mpox.

Source-supported facts: Mpox, earlier referred to as Monkeypox, is a viral disease of zoonotic origin with Smallpox-like symptoms but with less clinical severity. Since 1st January 2022, Mpox cases have been reported to WHO from 123 Member States across all 6 WHO regions. Mpox virus belongs to the genus Orthopoxvirus and family Poxviridae. Human-to-human transmission of Mpox occurs through large respiratory droplets requiring prolonged close contact. Complications of Mpox in children can be severe and may include secondary bacterial infections, severe skin infections, dehydration, pneumonia, encephalitis, ocular complications, and longer sequelae such as scarring of skin lesions.

Source: ncdc.mohfw.gov.in  
Licence: Not specified  
Dataset page: https://www.desidata.in/datasets/understanding-mpox-key-facts-and-global-impact

In [ ]:
!pip install pandas matplotlib --quiet

## Load the dataset

Dataset downloads require a free DD token. Create one in Profile & settings, then set `DD_TOKEN` in your environment.
In Colab, add `DD_TOKEN` under Secrets and enable notebook access to that secret.

In [ ]:
import io
import json
import os
import pandas as pd
import urllib.request

url = "https://www.desidata.in/api/datasets/understanding-mpox-key-facts-and-global-impact/download"
if "COLAB_RELEASE_TAG" in os.environ and "DD_TOKEN" not in os.environ:
    from google.colab import userdata
    os.environ["DD_TOKEN"] = userdata.get("DD_TOKEN")
request = urllib.request.Request(url, headers={"Authorization": f"Bearer {os.environ['DD_TOKEN']}", "Accept": "application/json"})
with urllib.request.urlopen(request) as response:
    file_url = json.load(response)["download_url"]
# Fetch the signed file URL separately; never send DD_TOKEN to the storage host.
with urllib.request.urlopen(file_url) as response:
    df = pd.read_csv(io.BytesIO(response.read()))

print(f"{df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

## Explore

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

## See one example

Every row pairs a question with the text it was written from.

In [ ]:
row = df.iloc[0]

print("QUESTION:", row["question"])

print("ANSWER:", str(row["answer"])[:400])

print()
print("SOURCE CONTEXT:")
print(str(row["context"])[:600])

## Mini baseline — can TF-IDF find the source paragraph?

Each question was written from its row's source paragraph. A simple TF-IDF
retriever has to rank the correct one first. Its hit rate is the number
any trained model on this dataset must beat.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
import numpy as np

sample = df.head(300)
questions = sample["question"].astype(str).to_numpy()
passages = sample["context"].astype(str).to_numpy()

vectorizer = TfidfVectorizer(stop_words="english")
passage_vectors = vectorizer.fit_transform(passages)
question_vectors = vectorizer.transform(questions)

best = np.empty(len(questions), dtype=int)
for i in range(len(questions)):
    best[i] = linear_kernel(question_vectors[i], passage_vectors).argmax()

hit_rate = (passages[best] == passages).mean()
print(f"TF-IDF retrieval lands on the correct paragraph for {hit_rate:.0%} of questions")
print("That is the number a trained model has to beat.")

## What will you build?

- Look for gaps across regions, age groups or demographics
- Build a simple risk score and validate it honestly on a hold-out set
- Chart how an indicator changes over time if the data has a year column

- Browse more [Health datasets](https://www.desidata.in/datasets?category=Health) or the [full catalogue](https://www.desidata.in/datasets).
- Sharing your work? Tag DesiData — we feature community notebooks built on this catalogue.

In [ ]:
# Data via DesiData — India's open dataset catalogue
print("Dataset page:", "https://www.desidata.in/datasets/understanding-mpox-key-facts-and-global-impact")